In [1]:
import findspark
findspark.init("/opt/manual/spark/")

In [2]:
from pyspark.sql import SparkSession, functions as F

In [3]:
spark = SparkSession.builder \
.appName("Simple ML Pipelines") \
.master("local[2]") \
.getOrCreate()

In [4]:
# ! wget -O ~/datasets/Mall_Customers.csv  https://raw.githubusercontent.com/erkansirin78/datasets/master/Mall_Customers.csv

In [5]:
df = spark.read.format("csv") \
.option("header",True) \
.option("inferSchema",True) \
.option("sep",",") \
.load("file:///home/train/datasets/Mall_Customers.csv") \
.withColumnRenamed("SpendingScore", "label")

In [6]:
df.show(5)

+----------+------+---+------------+-----+
|CustomerID|Gender|Age|AnnualIncome|label|
+----------+------+---+------------+-----+
|         1|  Male| 19|       15000|   39|
|         2|  Male| 21|       15000|   81|
|         3|Female| 20|       16000|    6|
|         4|Female| 23|       16000|   77|
|         5|Female| 31|       17000|   40|
+----------+------+---+------------+-----+
only showing top 5 rows



In [7]:
df.count()

200

In [8]:
df.groupBy("Gender").count().show()

+------+-----+
|Gender|count|
+------+-----+
|Female|  112|
|  Male|   88|
+------+-----+



# StringIndexer

In [9]:
from pyspark.ml.feature import StringIndexer

In [10]:
gender_stridx_obj = StringIndexer(inputCol="Gender", 
                                  outputCol="genderIdx", 
                                  handleInvalid='error')

# OneHotEncoder

In [11]:
from pyspark.ml.feature import OneHotEncoder

In [12]:
ohe_obj = OneHotEncoder(inputCols=["genderIdx"],
    outputCols=["OHE_Cols"],
    handleInvalid='error')

In [13]:
from pyspark.ml.feature import VectorAssembler

In [36]:
assembler_obj = VectorAssembler(inputCols=['Age','AnnualIncome','OHE_Cols'], 
                                outputCol='features', 
                                handleInvalid='skip')

In [37]:
from pyspark.ml.feature import StandardScaler

In [38]:
scaler = StandardScaler(withMean=False, 
                        withStd=True, 
                        inputCol='vector_assebmled_cols', 
                        outputCol='features')

# Split data

In [39]:
df_train, df_test = df.randomSplit([0.8, 0.2], seed=142)

In [40]:
print(df_train.count())
print(df_test.count())

161
39


# Estimator

In [41]:
from pyspark.ml.regression import DecisionTreeRegressor

In [42]:
estimator = DecisionTreeRegressor()

# Pipelines

In [25]:
from pyspark.ml.pipeline import Pipeline, PipelineModel

In [43]:
pipeline_obj = Pipeline().setStages([gender_stridx_obj, ohe_obj, assembler_obj, estimator])

# Train Model

In [44]:
pipeline_model = pipeline_obj.fit(df_train)

# Test Model

In [45]:
transformed_df = pipeline_model.transform(df_test)

In [46]:
transformed_df.limit(5).toPandas()

,CustomerID,Gender,Age,AnnualIncome,label,genderIdx,OHE_Cols,features,prediction
0,4,Female,23,16000,77,0.0,(1.0),"[23.0, 16000.0, 1.0]",76.866667
1,9,Male,64,19000,3,1.0,(0.0),"[64.0, 19000.0, 0.0]",4.000000
2,11,Male,67,19000,14,1.0,(0.0),"[67.0, 19000.0, 0.0]",4.000000
3,15,Male,37,20000,13,1.0,(0.0),"[37.0, 20000.0, 0.0]",54.421053
4,25,Female,54,28000,14,0.0,(1.0),"[54.0, 28000.0, 1.0]",16.571429


# Evaluate the model

In [47]:
from pyspark.ml.evaluation import RegressionEvaluator

In [48]:
evaluator = RegressionEvaluator(metricName='r2')

In [49]:
r2 = evaluator.evaluate(transformed_df)

In [50]:
print(r2)

0.34185668566090155


# Save Model

In [53]:
pipeline_model.write().overwrite().save("file:///home/train/atscale5/pyspark/08_ml/saved_models/simple_ml_pipeline")

# Load saved model

In [54]:
pipeline_model_loaded = PipelineModel.load("file:///home/train/atscale5/pyspark/08_ml/saved_models/simple_ml_pipeline")

# Predictions from loaded model

In [55]:
pipeline_model_loaded.transform(df_test).limit(5).toPandas()

,CustomerID,Gender,Age,AnnualIncome,label,genderIdx,OHE_Cols,features,prediction
0,4,Female,23,16000,77,0.0,(1.0),"[23.0, 16000.0, 1.0]",76.866667
1,9,Male,64,19000,3,1.0,(0.0),"[64.0, 19000.0, 0.0]",4.000000
2,11,Male,67,19000,14,1.0,(0.0),"[67.0, 19000.0, 0.0]",4.000000
3,15,Male,37,20000,13,1.0,(0.0),"[37.0, 20000.0, 0.0]",54.421053
4,25,Female,54,28000,14,0.0,(1.0),"[54.0, 28000.0, 1.0]",16.571429


# Just Save Regressor

In [58]:
type(pipeline_model)

pyspark.ml.pipeline.PipelineModel

In [59]:
type(pipeline_model.stages)

list

In [60]:
print(pipeline_model.stages)

[StringIndexerModel: uid=StringIndexer_1450cf6ec7f9, handleInvalid=error, OneHotEncoderModel: uid=OneHotEncoder_5ae9d48c9782, dropLast=true, handleInvalid=error, numInputCols=1, numOutputCols=1, VectorAssembler_c328636609f1, DecisionTreeRegressionModel: uid=DecisionTreeRegressor_7aa402d8600f, depth=5, numNodes=51, numFeatures=3]


In [61]:
type(pipeline_model.stages[-1])

pyspark.ml.regression.DecisionTreeRegressionModel

In [62]:
pipeline_model.stages[-1].write().overwrite().save("file:///home/train/atscale5/pyspark/08_ml/saved_models/simple_ml_regressor")

# Load Saved Regressor

In [64]:
loaded_regressor = DecisionTreeRegressionModel.loaded_regressor("file:///home/train/atscale5/pyspark/08_ml/saved_models/simple_ml_regressor")

NameError: name 'DecisionTreeRegressionModel' is not defined

In [67]:
spark.stop()